In [ ]:
# # ============================================================
# # CELL 0: INSTALL ALL REQUIRED PACKAGES (run once, then restart kernel)
# # ============================================================
# import subprocess
# import sys

# # Uninstall any existing CPU-only torch build first
# subprocess.run([sys.executable, "-m", "pip", "uninstall", "torch", "torchvision", "torchaudio", "-y"])

# # Install CUDA-enabled PyTorch
# subprocess.run([
#     sys.executable, "-m", "pip", "install", "torch", "torchvision",
#     "--index-url", "https://download.pytorch.org/whl/cu121"
# ])

# # Install remaining packages used across the notebook
# subprocess.run([
#     sys.executable, "-m", "pip", "install",
#     "scikit-learn", "numpy", "matplotlib", "pillow", "jupyterlab", "tqdm"
# ])

# print("\nDone. RESTART THE KERNEL now (Kernel -> Restart) before running any other cells.")

In [ ]:
# ============================================================
# CELL 1: CONFIG — change MODEL_NAME / TRAIN_MODE here per run
# ============================================================
import os, time, json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "Medicinal Plant Leaf Health Split Dataset"   # SAME for all models — do not change per model
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 30
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "efficientnet_b0"   # mobilenet_v3_small | shufflenet_v2 | efficientnet_b0
TRAIN_MODE = "partial"                  # full | partial | classifier_only
PARTIAL_UNFREEZE_LAST_N = 2

# each model+mode combo writes to its OWN folder automatically -> safe to rerun for all combos
RUN_TAG = f"{MODEL_NAME}_{TRAIN_MODE}"
SAVE_DIR = os.path.join("checkpoints", RUN_TAG)
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT_PATH = os.path.join(SAVE_DIR, "last_checkpoint.pth")
BEST_PATH = os.path.join(SAVE_DIR, "best_model.pth")
RESULTS_PATH = os.path.join(SAVE_DIR, "results.json")
CURVES_PATH = os.path.join(SAVE_DIR, "curves.png")
HISTORY_PATH = os.path.join(SAVE_DIR, "history.json")

print(f"Data source (shared across all runs): {DATA_DIR}")
print(f"This run's outputs will be saved to: {SAVE_DIR}")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: No GPU detected — training will run on CPU and be very slow.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# ============================================================
# CELL 2: TRANSFORMS — train gets augmentation, val/test don't
# ============================================================
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# ============================================================
# CELL 3: LOAD DATA — reads from your existing train/val/test folders
# Verifies each split's 16 class subfolders and image counts
# ============================================================
train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=eval_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

NUM_CLASSES = len(train_ds.classes)
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")
print(f"Train images: {len(train_ds)} | Val images: {len(val_ds)} | Test images: {len(test_ds)}")

# sanity check: same 16 classes in every split, in the same order
assert train_ds.classes == val_ds.classes == test_ds.classes, "Class folder mismatch between splits!"
print("Class folders match across train/val/test ✓")

# ============================================================
# AUGMENTATION COUNT INFO
# ============================================================
print(f"\nOriginal training images (on disk): {len(train_ds)}")
print("Note: augmentation here is applied ON-THE-FLY (random transforms per epoch),")
print("so no new image files are created — each epoch sees the same 1,323-based split,")
print("but each training image gets a randomly different augmented version every epoch.")
print(f"Effective images seen per training epoch: {len(train_ds)} (same count, transformed each time)")
print(f"Effective images seen across all {EPOCHS} epochs: {len(train_ds) * EPOCHS}")

In [ ]:
# ============================================================
# CELL 4: CLASS WEIGHTS — compensates for imbalance (52–153 per class)
# ============================================================
class_counts = np.bincount([label for _, label in train_ds.samples])
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Per-class training image counts:", class_counts)
print("Computed class weights:", class_weights.cpu().numpy())

In [ ]:
# ============================================================
# CELL 5: MODEL BUILDER — swaps in MobileNetV3 / ShuffleNetV2 / EfficientNet-B0
# ============================================================
def build_model(name, num_classes):
    if name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights="IMAGENET1K_V1")
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
        feature_module = m.features
    elif name == "shufflenet_v2":
        m = models.shufflenet_v2_x1_0(weights="IMAGENET1K_V1")
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        feature_module = nn.Sequential(m.conv1, m.maxpool, m.stage2, m.stage3, m.stage4, m.conv5)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights="IMAGENET1K_V1")
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        feature_module = m.features
    else:
        raise ValueError(f"Unknown model: {name}")
    return m, feature_module

In [ ]:
# ============================================================
# CELL 6: FREEZE/UNFREEZE LOGIC — controls full / partial / classifier_only
# ============================================================
def apply_train_mode(model, feature_module, mode, unfreeze_last_n=2):
    for p in model.parameters():
        p.requires_grad = False

    if mode == "full":
        for p in model.parameters():
            p.requires_grad = True
    elif mode == "classifier_only":
        for name, p in model.named_parameters():
            if "classifier" in name or name.startswith("fc."):
                p.requires_grad = True
    elif mode == "partial":
        children = list(feature_module.children())
        unfreeze_blocks = children[-unfreeze_last_n:] if unfreeze_last_n > 0 else []
        for block in unfreeze_blocks:
            for p in block.parameters():
                p.requires_grad = True
        for name, p in model.named_parameters():
            if "classifier" in name or name.startswith("fc."):
                p.requires_grad = True
    else:
        raise ValueError(f"Unknown TRAIN_MODE: {mode}")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[{mode}] Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model

model, feature_module = build_model(MODEL_NAME, NUM_CLASSES)
model = apply_train_mode(model, feature_module, TRAIN_MODE, PARTIAL_UNFREEZE_LAST_N)
model = model.to(DEVICE)

In [ ]:
# ============================================================
# CELL 7: LOSS / OPTIMIZER — only trainable params passed to optimizer
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)

In [ ]:
# ============================================================
# CELL 8: EPOCH RUNNER FUNCTION — one pass over a loader (train or eval)
# ============================================================
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1, precision, recall

In [ ]:
# ============================================================
# CELL 9: CHECKPOINT RESUME — auto-continues if this cell reruns after interruption
# ============================================================
start_epoch = 1
best_val_f1 = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}

if os.path.exists(CKPT_PATH):
    print(f"Resuming from checkpoint: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_f1 = ckpt["best_val_f1"]
    if os.path.exists(HISTORY_PATH):
        with open(HISTORY_PATH) as f:
            history = json.load(f)
    print(f"Resumed at epoch {start_epoch}, best_val_f1={best_val_f1:.4f}")

In [ ]:
# ============================================================
# CELL 10: TRAINING LOOP — trains, saves checkpoint + best model every epoch
# ============================================================
from tqdm.notebook import tqdm

def run_epoch(loader, train=True, desc="Epoch"):
    model.train() if train else model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    context = torch.enable_grad() if train else torch.no_grad()
    pbar = tqdm(loader, desc=desc, leave=False)
    with context:
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, macro_f1, precision, recall


for epoch in range(start_epoch, EPOCHS + 1):
    train_loss, train_acc, train_f1, _, _ = run_epoch(train_loader, train=True, desc=f"Epoch {epoch}/{EPOCHS} [train]")
    val_loss, val_acc, val_f1, val_prec, val_rec = run_epoch(val_loader, train=False, desc=f"Epoch {epoch}/{EPOCHS} [val]")
    scheduler.step(val_f1)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    print(f"[{RUN_TAG}] Epoch {epoch}/{EPOCHS} | "
          f"Train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}", flush=True)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  -> New best model saved (val macro-F1={val_f1:.4f})", flush=True)

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_f1": best_val_f1,
    }, CKPT_PATH)

    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=2)

In [ ]:
# ============================================================
# CELL 11: PLOT CURVES — loss / accuracy / macro-F1, train vs val
# ============================================================
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, history["train_loss"], label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val Loss")
axes[0].set_title(f"{RUN_TAG} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train Acc")
axes[1].plot(epochs_range, history["val_acc"], label="Val Acc")
axes[1].set_title(f"{RUN_TAG} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(epochs_range, history["train_f1"], label="Train Macro-F1")
axes[2].plot(epochs_range, history["val_f1"], label="Val Macro-F1")
axes[2].set_title(f"{RUN_TAG} - Macro-F1"); axes[2].set_xlabel("Epoch"); axes[2].legend()

plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=300)
plt.show()
print(f"Saved curves to {CURVES_PATH}")

In [ ]:
# ============================================================
# CELL 12: FINAL TEST EVAL — loads best checkpoint, measures inference speed
# ============================================================
model.load_state_dict(torch.load(BEST_PATH))
model.eval()

sample_batch, _ = next(iter(test_loader))
sample_batch = sample_batch.to(DEVICE)
with torch.no_grad():
    start = time.time()
    for _ in range(20):
        _ = model(sample_batch)
    elapsed = time.time() - start
per_image_ms = (elapsed / (20 * sample_batch.size(0))) * 1000

test_loss, test_acc, test_f1, test_prec, test_rec = run_epoch(test_loader, train=False)

num_params = sum(p.numel() for p in model.parameters())
trainable_params_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = num_params * 4 / (1024 ** 2)

results = {
    "model": MODEL_NAME, "train_mode": TRAIN_MODE,
    "test_accuracy": test_acc, "test_macro_f1": test_f1,
    "test_precision": test_prec, "test_recall": test_rec,
    "total_params": num_params, "trainable_params": trainable_params_count,
    "model_size_mb": round(model_size_mb, 3),
    "inference_ms_per_image": round(per_image_ms, 3),
    "best_val_macro_f1": best_val_f1,
}
# ============================================================
# CONFUSION MATRIX
# ============================================================
from sklearn.metrics import confusion_matrix
import seaborn as sns

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
class_names = test_ds.classes

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"{RUN_TAG} - Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

cm_path = os.path.join(SAVE_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.show()
print(f"Saved confusion matrix to {cm_path}")

print("\n=== FINAL TEST RESULTS ===")
print(json.dumps(results, indent=2))

with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
#run cell 1-3 before running bellow cell seperately if not let it as it is
# ============================================================
# STANDALONE: LOAD MODEL + RUN TEST PREDICTIONS LATER
# ============================================================
import os
import torch
import torch.nn as nn
import pandas as pd
from torchvision import models

# ---- CONFIG: pick which run's results you want ----
MODEL_NAME = "efficientnet_b0"   # mobilenet_v3_small | shufflenet_v2 | efficientnet_b0
TRAIN_MODE = "partial"                  # full | partial | classifier_only

RUN_TAG = f"{MODEL_NAME}_{TRAIN_MODE}"
SAVE_DIR = os.path.join("checkpoints", RUN_TAG)
BEST_PATH = os.path.join(SAVE_DIR, "best_model.pth")

assert os.path.exists(BEST_PATH), f"No saved model found at {BEST_PATH} — train this combo first."

# ---- Rebuild architecture (must match training) ----
def build_model(name, num_classes):
    if name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif name == "shufflenet_v2":
        m = models.shufflenet_v2_x1_0(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m

NUM_CLASSES = len(test_ds.classes)  # assumes test_ds already loaded from Cell 3
model = build_model(MODEL_NAME, NUM_CLASSES)
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print(f"Loaded best model for {RUN_TAG} from {BEST_PATH}")

# ---- Run test predictions ----
class_names = test_ds.classes
test_samples = test_ds.samples
prediction_log = []

idx = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)
        confidences = probs.max(dim=1).values

        for i in range(imgs.size(0)):
            filepath, true_idx = test_samples[idx]
            pred_idx = preds[i].item()
            prediction_log.append({
                "image_path": filepath,
                "true_class": class_names[true_idx],
                "predicted_class": class_names[pred_idx],
                "confidence": round(confidences[i].item(), 4),
                "correct": true_idx == pred_idx,
            })
            idx += 1

pred_log_df = pd.DataFrame(prediction_log)
pred_log_path = os.path.join(SAVE_DIR, "test_predictions_log.csv")
pred_log_df.to_csv(pred_log_path, index=False)

print(f"Saved per-image test predictions to {pred_log_path}")
print(f"Total test images: {len(pred_log_df)}")
print(f"Correct: {pred_log_df['correct'].sum()} | Incorrect: {(~pred_log_df['correct']).sum()}")
print("\n=== Sample of misclassified images ===")
print(pred_log_df[~pred_log_df['correct']].head(10).to_string(index=False))

In [ ]:
# ============================================================
# PER-CLASS RESULTS (from standalone prediction run)
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_true = pred_log_df["true_class"]
y_pred = pred_log_df["predicted_class"]

# per-class precision/recall/f1
report_dict = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().round(4)
print("=== PER-CLASS RESULTS (Test Set) ===")
print(report_df)

report_csv_path = os.path.join(SAVE_DIR, "per_class_report.csv")
report_df.to_csv(report_csv_path)
print(f"Saved per-class report to {report_csv_path}")

# per-class F1 bar chart
per_class_f1 = report_df.iloc[:len(class_names)]["f1-score"]
plt.figure(figsize=(12, 6))
per_class_f1.sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("F1-Score")
plt.title(f"{RUN_TAG} - Per-Class F1 (Test Set)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "per_class_f1.png"), dpi=300)
plt.show()

# confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=class_names)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"{RUN_TAG} - Confusion Matrix")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=300)
plt.show()

# top misclassification per class
print("\n=== TOP MISCLASSIFICATIONS PER CLASS ===")
misclass_rows = []
for i, true_cls in enumerate(class_names):
    row = cm[i].copy()
    correct = row[i]
    row[i] = 0
    total = cm[i].sum()
    if row.sum() > 0:
        top_idx = row.argmax()
        misclass_rows.append({
            "true_class": true_cls, "correct_predictions": correct, "total_samples": total,
            "most_confused_with": class_names[top_idx], "confused_count": row[top_idx],
        })
        print(f"{true_cls}: {correct}/{total} correct | most confused with '{class_names[top_idx]}' ({row[top_idx]} times)")
    else:
        misclass_rows.append({
            "true_class": true_cls, "correct_predictions": correct, "total_samples": total,
            "most_confused_with": "None (perfect)", "confused_count": 0,
        })
        print(f"{true_cls}: {correct}/{total} correct | no misclassifications")

misclass_df = pd.DataFrame(misclass_rows)
misclass_df.to_csv(os.path.join(SAVE_DIR, "misclassification_summary.csv"), index=False)
print(f"\nSaved misclassification summary to {os.path.join(SAVE_DIR, 'misclassification_summary.csv')}")